In [4]:
'''관련 패키지 import'''
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys

from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

import requests

import getpass

import pandas as pd

import time

from tabulate import tabulate

In [51]:
# 1. 크롬 드라이버 경로 설정
chromedriver_path = './chromedriver.exe'
service = Service(executable_path=chromedriver_path)

# 2. 웹 브라우저 열기 (Chrome)
driver = webdriver.Chrome(service=service)
driver.maximize_window()
print("크롬 브라우저 실행 완료!")

# 3. 특정 URL로 이동
url = "https://www.snulife.com/lecture"
driver.get(url)
print(f"{url}로 이동 완료!")


try:
    wait = WebDriverWait(driver, 15) # 5초 대기 설정

    # 4. 로그인
    id_box = wait.until(EC.element_to_be_clickable(
        (By.XPATH, "//*[@id='__next']/div/div[2]/div[2]/div[1]/div[2]/div[1]/div/form/div[1]/div/input[1]")
    ))
    pw_box = driver.find_element(By.XPATH, "//*[@id='__next']/div/div[2]/div[2]/div[1]/div[2]/div[1]/div/form/div[1]/div/input[2]")

    # ID = getpass.getpass("ID : ")
    # PW = getpass.getpass("PW : ")
    ID = 'dlaekdna4862'
    PW = 'ekdna7913'

    # ID, PW 입력
    id_box.send_keys(ID)
    pw_box.send_keys(PW)

    # ENTER 키 입력
    pw_box.send_keys(Keys.ENTER)

    # 로그인 완료할 때까지 기다리기!
    wait.until(EC.staleness_of(pw_box)) # 추후, 로그인 실패 로직 짜기!

    print("로그인 성공!")

    # 5. 교과목 검색
    search_box = wait.until(EC.element_to_be_clickable(
        (By.XPATH, '//*[@id="__next"]/div/div[1]/div/div/div[1]/div/div/input')
    ))

    # title = input("검색하고자 하는 교과목의 이름을 알려주세요. :") # 추후, 프롬프트에서 title을 추출해 param으로 전달 받도록 변경 필요.
    title = '경영 과학 1'
    title = title.replace(" ", "")

    # 교과목 이름 입력
    search_box.send_keys(f"{title}")

    # ENTER 키 입력
    search_box.send_keys(Keys.ENTER)

    wait.until(
        lambda driver: driver.find_element(By.XPATH, '//*[@id="__next"]/div/div[1]/div/div/div[1]/div/div/input').get_attribute("value")
    )

    # 6. 교과목 이름, 교수 이름 -> 적합한 교과목 링크 접속
    # prof_name = input("교과목의 담당 교수의 이름을 알려주세요. :") # 추후, 프롬프트에서 prof_name을 추출해 param으로 전달 받도록 변경 필요.
    prof_name = '홍성필'

    time.sleep(3)

    a_xpath = '//*[@id="__next"]/div/div[2]/div/div[1]/div[3]'
    a_elem = wait.until(EC.visibility_of_element_located(
        (By.XPATH, a_xpath)
    ))
    a_elem_num = len(a_elem.find_elements(By.XPATH, './a'))

    data = []

    for i in range(1, a_elem_num+1):
        tmp_xpath = f'//*[@id="__next"]/div/div[2]/div/div[1]/div[3]/a[{i}]'

        tmp_title = driver.find_element(By.XPATH, tmp_xpath + '/div[1]/div[2]').text
        tmp_title = tmp_title.replace(" ", "")

        if tmp_title == title:
            tmp_prof_name = driver.find_element(By.XPATH, tmp_xpath + '/div[2]/div[1]/span[1]').text
            
            if tmp_prof_name == prof_name:
                tmp_class = driver.find_element(By.XPATH, tmp_xpath + '/div[1]/div[1]').text
                tmp_dep = driver.find_element(By.XPATH, tmp_xpath + '/div[2]/div[1]/span[3]').text
                tmp_rating = driver.find_element(By.XPATH, tmp_xpath + '/div[1]/div[3]').text
                tmp_review_num = driver.find_element(By.XPATH, tmp_xpath + '/div[2]/div[2]/span[1]').text[4:]+"개"
                tmp_prev_exam = driver.find_element(By.XPATH, tmp_xpath + '/div[2]/div[2]/span[3]').text[3:]+"개"

                data.append([i, tmp_class, tmp_dep, tmp_rating, tmp_review_num, tmp_prev_exam])

    if data:
        headers = ['INDEX', '교과 구분', '학과', '별점(10점 만점)', '강의평', '족보']
        col_widths = [10, 10, 20, 10, 10, 10]
        
        header_line = ""
        for i, header in enumerate(headers):
            header_line += f"{header:^{col_widths[i]}}"
        print(header_line)
        print("-"*sum(col_widths))

        for row in data:
            row_line = ""
            for i, item in enumerate(row):
                row_line += f"{item:^{col_widths[i]}}"
            print(row_line)

    else:
        print("조건에 맞는 강의가 없음!")
                

except Exception as e:
    print(f"{e}")

크롬 브라우저 실행 완료!
https://www.snulife.com/lecture로 이동 완료!
로그인 성공!
  INDEX     교과 구분            학과         별점(10점 만점)   강의평        족보    
----------------------------------------------------------------------
    1         전필           산업공학과            1         1개        3개    
    3         전필           산업공학과           7.6       52개        4개    
    7         전필       산업공학과(산업공학전공)        9         3개        0개    


In [52]:
# 몇 번째에 링크에 접속할 지 결정되었다고 가정
idx = 3
decided_xpath = f'//*[@id="__next"]/div/div[2]/div/div[1]/div[3]/a[{idx}]'
decided_lecture = driver.find_element(By.XPATH, decided_xpath)
decided_lecture.send_keys(Keys.ENTER)


In [53]:
# 강의평 -> 리스트 형식으로 받아오기
reviews = []

review_xpath = driver.find_element(By.XPATH, '//*[@id="__next"]/div/div[2]/div[5]/div')
reviews_num = len(review_xpath.find_elements(By.XPATH, './div'))

for i in range(1, reviews_num+1):
    tmp_xpath = f'//*[@id="__next"]/div/div[2]/div[5]/div/div[{i}]/div/div[3]'
    tmp_review = driver.find_element(By.XPATH, tmp_xpath).text

    if tmp_review:
        reviews.append(tmp_review)

In [54]:
# 족보 -> 다운로드 버튼 누르기
족보_button = driver.find_element(By.XPATH, '//*[@id="__next"]/div/div[2]/div[4]/div/button[2]')
족보_button.send_keys(Keys.ENTER)

time.sleep(2)

download_xpath = driver.find_element(By.XPATH, '//*[@id="__next"]/div/div[2]/div[5]/div')
downloads_num = len(download_xpath.find_elements(By.XPATH, './div'))

In [ ]:
for i in range(1, downloads_num+1):
    tmp_download_button = driver.find_element(By.XPATH, f'//*[@id="__next"]/div/div[2]/div[5]/div/div[{i}]/div[4]/a/button/span')
    tmp_download_button.send_keys(Keys.ENTER)
    time.sleep(2)

# 이거 새 창이 띄워진다!

ElementNotInteractableException: Message: element not interactable
  (Session info: chrome=142.0.7444.60); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#elementnotinteractableexception
Stacktrace:
Symbols not available. Dumping unresolved backtrace:
	0x7ff6b9f37a35
	0x7ff6b9f37a90
	0x7ff6b9cb14b5
	0x7ff6b9d038e3
	0x7ff6b9d014c6
	0x7ff6b9d3297a
	0x7ff6b9cfcb56
	0x7ff6b9d5b8fb
	0x7ff6b9cfb068
	0x7ff6b9cfbe93
	0x7ff6ba1f29d0
	0x7ff6ba1ece50
	0x7ff6ba20cc45
	0x7ff6b9f530ce
	0x7ff6b9f5adbf
	0x7ff6b9f40c14
	0x7ff6b9f40dcf
	0x7ff6b9f26828
	0x7ffdf5afe8d7
	0x7ffdf724c53c
